# MNIST FPGA Live Cloud Classification

This notebook documents the **final live FPGA-to-Cloud Run MNIST classification implementation**.

Unlike the formal cloud-measurement project, this version does not use controlled 100-image batches or store measurement-result records. Instead, the FPGA captures and freezes one camera-derived 28 × 28 image, transfers the 784 pixels to the Zynq Processing System through the shared AXI BRAM mailbox, sends the image to Google Cloud Run, receives the predicted digit, and displays the returned prediction on the ZedBoard LEDs.


## 1. Live cloud inference path

```text
OV7670 camera
    ↓
ROI capture / downsampling
    ↓
frozen 28 × 28 image in BRAM
    ↓
784-byte AXI BRAM mailbox transfer
    ↓
Zynq Processing System
    ↓
lwIP / TCP / Mbed TLS
    ↓
HTTPS POST /predict
    ↓
Google Cloud Run MNIST service
    ↓
JSON prediction
    ↓
prediction written to mailbox
    ↓
PL latches returned digit
    ↓
prediction displayed on LEDs
```

This branch represents the final functional camera-to-cloud path rather than the earlier formal measurement configuration.


## 2. Live implementation evidence

The repository contains:

- one live-inference video;
- nine photographs from the live-cloud implementation;
- placed-utilization report;
- routed timing-summary report;
- route-status report; and
- routed Vivado power report.

The live-inference video is:

[Cloud Live Inference10-21-202613-03.mp4](<Cloud Live Inference10-21-202613-03.mp4>)


## 3. Physical live-cloud setup

The following photograph shows the complete live setup with the camera enclosure, monitor, ZedBoard, Ethernet connection, and frozen camera-derived digit displayed on the monitor.

![FPGA live cloud inference overview](FPGA_Live_Cloud_Inference_Overview.jpg)


## 4. Physical power observation

The repository also includes a photograph of the Kill A Watt meter during the live-cloud configuration.

![FPGA live cloud power](FPGA_Live_Cloud_Power.jpg)

The physical meter observation represents the complete local system boundary. It does not include electrical power consumed by the remote Google Cloud Run infrastructure.


## 5. Additional photographs

The repository includes the remaining live-cloud photographs as supporting evidence of the implementation, frozen-image workflow, returned prediction, and board state.

The following cell can be run after cloning the repository to list the image files exactly as stored.


In [ ]:
from pathlib import Path

image_files = sorted(
    list(Path(".").glob("*.jpg")) +
    list(Path(".").glob("*.jpeg")) +
    list(Path(".").glob("*.png"))
)

print(f"Image files found: {len(image_files)}")
for f in image_files:
    print(f.name)


## 6. Live mailbox interface

The stripped live-cloud SDK uses only the mailbox fields needed for one frozen image and one returned prediction:

```text
0x000–0x30C   784 image bytes packed four per 32-bit word
0x310         STATUS
0x314         IMAGE_INDEX
0x318         RESULT
0x31C         EXPECTED field; not applicable to live camera input
```

Handshake values:

```text
0 = idle
1 = image ready
2 = result ready
```

The formal cloud-measurement fields for accuracy summaries, per-image records, total-run timing, and `TEST_COMPLETE` are not part of this live deployment path.


## 7. SDK live-client behavior

The live SDK application performs the same network bring-up developed during the cloud-measurement work:

```text
platform initialization
    ↓
lwIP initialization
    ↓
Ethernet / DHCP
    ↓
DNS
    ↓
TCP connection
    ↓
Mbed TLS
    ↓
TLS handshake
    ↓
wait for frozen image
```

Once `STATUS = IMAGE_READY`, the PS reads 196 32-bit mailbox words and unpacks the 784 image bytes.

The request uses:

```text
POST /predict HTTP/1.1
Content-Type: application/octet-stream
Content-Length: 784
Connection: keep-alive
```

The returned JSON prediction is parsed and written to `MAILBOX_RESULT` before the PS sets `STATUS = RESULT_READY`.


## 8. Returned prediction and LED display

The live implementation removes the formal result-dump/storage path.

After Cloud Run returns the prediction:

```text
PS writes RESULT
    ↓
PS writes STATUS = RESULT_READY
    ↓
PL detects valid result
    ↓
PL latches prediction
    ↓
prediction appears on ZedBoard LEDs
```

This provides a direct physical indication of the cloud-classified frozen image.


## 9. Live diagnostic timing

The live SDK client can print the returned prediction and request latency over UART while validating the camera-to-cloud path.

This timing is a **functional live diagnostic** and is not part of the formal 100-image cloud latency dataset.

The formal cloud latency and throughput results are documented in the cloud-measurement branches.


## 10. Vivado reports

The live Notebook-and-Data branch includes:

```text
design_1_wrapper_utilization_placed_cloud_live.rpt
design_1_wrapper_timing_summary_routed_live_cloud.rpt
design_1_wrapper_route_status_live_cloud.rpt
design_1_wrapper_power_routed_live_cloud.rpt
```

These reports document the final live-cloud FPGA configuration rather than the earlier formal cloud-measurement build.


### Report extraction

The following cell reads the report files directly after cloning the repository and prints key utilization, timing, routing, and power lines.


In [ ]:
from pathlib import Path
import re

report_files = {
    "utilization": Path("design_1_wrapper_utilization_placed_cloud_live.rpt"),
    "timing": Path("design_1_wrapper_timing_summary_routed_live_cloud.rpt"),
    "routing": Path("design_1_wrapper_route_status_live_cloud.rpt"),
    "power": Path("design_1_wrapper_power_routed_live_cloud.rpt"),
}

patterns = {
    "utilization": [
        r"Slice LUTs.*",
        r"Slice Registers.*",
        r"Block RAM Tile.*",
        r"DSPs.*",
    ],
    "timing": [
        r".*WNS.*TNS.*",
        r".*Design Timing Summary.*",
    ],
    "routing": [
        r".*routable nets.*",
        r".*fully routed.*",
        r".*routing errors.*",
    ],
    "power": [
        r".*Total On-Chip Power.*",
        r".*Dynamic \(W\).*",
        r".*Device Static \(W\).*",
        r".*Confidence Level.*",
    ],
}

for key, path in report_files.items():
    print(f"\\n=== {key.upper()} ===")
    if not path.exists():
        print(f"Missing: {path.name}")
        continue

    text = path.read_text(errors="ignore").splitlines()

    found = []
    for line in text:
        for pattern in patterns[key]:
            if re.search(pattern, line, re.IGNORECASE):
                if line.strip() not in found:
                    found.append(line.strip())

    if found:
        for line in found:
            print(line)
    else:
        print("No matching summary lines found; inspect report manually.")


## 11. Vivado routed power

The routed live-cloud power report records a Vivado implementation estimate of:

```text
Total on-chip power: 1.769 W
Dynamic power:       1.621 W
Device static power: 0.148 W
Confidence level:    Low
```

The low-confidence rating indicates that the Vivado estimate should be treated as an implementation estimate rather than a substitute for the physical power-meter evidence.


## 12. Difference from formal cloud measurement

The formal cloud-measurement implementation used:

```text
controlled MNIST ROM
100-image sequence
expected labels
accuracy counters
per-image latency records
run-summary storage
TEST_COMPLETE
```

The final live-cloud implementation instead uses:

```text
frozen camera image
one live request at a time
returned cloud prediction
LED output
```

The removal of the measurement-storage structures makes this branch the final functional live-deployment path.


## 13. Difference from local live inference

The earlier local live implementation classified the frozen image with the ZyNet inference engine on the FPGA.

The live cloud version replaces local classification with:

```text
frozen FPGA image
    ↓
Zynq PS
    ↓
Cloud Run
    ↓
returned prediction
```

The prediction is still presented physically on the ZedBoard LEDs, allowing the live local and live cloud implementations to use a comparable visible output while differing in where inference executes.


## 14. Supporting code branch

The corresponding live-cloud code branch contains:

```text
top_live_cloud.vhd
design_1_bd_live.tcl
cloud_live.bit
main.c
tcp_client.c
```

The code branch preserves the implementation. This Notebook-and-Data branch preserves the implementation reports and visual evidence.


## 15. Scope

This notebook documents the final **live camera-to-cloud MNIST implementation**.

It does not reproduce the formal 1,000-image accuracy experiment or the formal 100-image latency/throughput runs. Those measurements belong to the separate cloud-measurement branches.

The purpose of the live branch is to demonstrate that a real camera-derived image can be frozen on the FPGA, transmitted through the Zynq PS to Cloud Run, classified remotely, returned to the FPGA, and displayed on the physical board.
